<div style="background:#0A2540;color:white;padding:40px 32px;border-radius:12px;">
<h1 style="color:#4FC3F7;margin:0 0 8px 0;">🇧🇷 O Brasil no Mapa da IA</h1>
<h2 style="color:white;font-weight:300;margin:0 0 24px 0;">Como usamos o Claude — e o que os dados dizem sobre hype vs. realidade</h2>
<p style="color:#B0BEC5;margin:0;">Fonte: Anthropic Economic Index · 3 releases (Ago 2025 → Fev 2026) · Análise: Abril 2026</p>
</div>

---
## Setup

In [ ]:
%pip install -q huggingface_hub pandas matplotlib seaborn plotly

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path
from huggingface_hub import hf_hub_download, list_repo_files

DATA_DIR   = Path('../data')
OUTPUT_DIR = Path('../outputs')
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 130
TITLE_SIZE = 15

BRAZIL      = 'BRA'
COMPARABLES = ['BRA', 'ARG', 'MEX', 'IND', 'ZAF', 'COL', 'CHL']
LABELS      = {
    'BRA': 'Brasil',        'ARG': 'Argentina',
    'MEX': 'México',        'IND': 'Índia',
    'ZAF': 'África do Sul', 'COL': 'Colômbia',
    'CHL': 'Chile'
}

REPO = 'Anthropic/EconomicIndex'

def dl(filename):
    return Path(hf_hub_download(repo_id=REPO, filename=filename,
                                repo_type='dataset', local_dir=DATA_DIR))

print('Setup concluído.')

---
## Descoberta das releases disponíveis

Usamos `list_repo_files` para mapear todos os arquivos AEI do repositório — enriquecidos e raw — antes de baixar.

In [ ]:
all_files = list(list_repo_files(REPO, repo_type='dataset'))

# Arquivos AEI com dados geográficos (enriched ou raw claude_ai)
aei_files = [
    f for f in all_files
    if ('aei_enriched_claude_ai' in f or 'aei_raw_claude_ai' in f)
    and f.endswith('.csv')
]

print(f'{len(aei_files)} arquivo(s) AEI claude_ai encontrado(s):\n')
for f in sorted(aei_files):
    print(f'  {f}')

---
## Carregando todas as releases

Três snapshots com dados geográficos reais:

| Release | Arquivo | Período dos dados | Schema |
|---------|---------|-------------------|--------|
| `release_2025_09_15` | `aei_enriched_claude_ai_…` | 04–11 Ago 2025 | **Enriquecido** — tem `usage_per_capita_index` |
| `release_2026_01_15` | `aei_raw_claude_ai_…` | 13–20 Nov 2025 | Raw — `usage_pct` + normalização manual |
| `release_2026_03_24` | `aei_raw_claude_ai_…` | 05–12 Fev 2026 | Raw — `usage_pct` + normalização manual |

> Para comparar o índice per capita nas releases raw, calculamos:
> `usage_per_capita_index = usage_pct / (pop_country / pop_total_dataset)`

In [ ]:
# Mapeamento fixo: release → arquivo + data de referência (ponto médio da semana)
RELEASES = [
    {
        'release':      'release_2025_09_15',
        'file':         'release_2025_09_15/data/output/aei_enriched_claude_ai_2025-08-04_to_2025-08-11.csv',
        'reference_dt': pd.Timestamp('2025-08-07'),
        'label':        'Ago 2025',
        'enriched':     True,
    },
    {
        'release':      'release_2026_01_15',
        'file':         'release_2026_01_15/data/intermediate/aei_raw_claude_ai_2025-11-13_to_2025-11-20.csv',
        'reference_dt': pd.Timestamp('2025-11-16'),
        'label':        'Nov 2025',
        'enriched':     False,
    },
    {
        'release':      'release_2026_03_24',
        'file':         'release_2026_03_24/data/aei_raw_claude_ai_2026-02-05_to_2026-02-12.csv',
        'reference_dt': pd.Timestamp('2026-02-08'),
        'label':        'Fev 2026',
        'enriched':     False,
    },
]

# População por país (usada para normalizar as releases raw)
df_pop = pd.read_csv(dl('release_2025_09_15/data/input/working_age_pop_2024_country_raw.csv'))
iso_col = next(c for c in df_pop.columns if 'alpha' in c.lower() or 'iso' in c.lower())
pop_col = next(c for c in df_pop.columns if 'pop' in c.lower())
pop_map  = df_pop.set_index(iso_col)[pop_col].to_dict()   # iso3 → working_age_pop
total_pop_in_data = sum(pop_map.values())
pop_share = {k: v / total_pop_in_data for k, v in pop_map.items()}

print(f'Populações mapeadas: {len(pop_map)} países')

frames = []
for r in RELEASES:
    print(f"\nBaixando {r['label']} …")
    raw = pd.read_csv(dl(r['file']))
    raw['release_date']  = r['reference_dt']
    raw['release_label'] = r['label']

    # Se for arquivo raw, calcula usage_per_capita_index a partir de usage_pct
    if not r['enriched']:
        usage_pct_rows = raw[
            (raw['geography'] == 'country') &
            (raw['facet']     == 'country') &
            (raw['variable']  == 'usage_pct')
        ].copy()
        usage_pct_rows['value'] = usage_pct_rows.apply(
            lambda row: row['value'] / pop_share.get(row['geo_id'], float('nan')),
            axis=1
        )
        usage_pct_rows['variable'] = 'usage_per_capita_index'
        raw = pd.concat([raw, usage_pct_rows], ignore_index=True)

    frames.append(raw)
    print(f"  {len(raw):,} linhas — variáveis: {sorted(raw['variable'].unique())[:6]} …")

df_all = pd.concat(frames, ignore_index=True)
print(f'\nDataframe combinado: {df_all.shape}')
print('Releases:', df_all['release_label'].unique())

In [ ]:
# Carrega release principal (Ago 2025) para os slides de corte transversal
df = df_all[df_all['release_label'] == 'Ago 2025'].copy()
df_job = pd.read_csv(dl('labor_market_impacts/job_exposure.csv'))

print('Dados prontos para análise de corte transversal (Ago 2025) + temporal (3 releases).')

---
# 📈 Slide 0 — O Brasil está crescendo ou estagnando?

**Pergunta:** A adoção de IA no Brasil evoluiu entre Agosto de 2025 e Fevereiro de 2026?

In [ ]:
df_trend = (
    df_all[
        (df_all['geography'] == 'country') &
        (df_all['facet']     == 'country') &
        (df_all['variable']  == 'usage_per_capita_index') &
        (df_all['geo_id'].isin(COMPARABLES))
    ]
    .assign(country=lambda x: x['geo_id'].map(LABELS))
    .sort_values('release_date')
)

fig, ax = plt.subplots(figsize=(11, 5))

for geo_id, group in df_trend.groupby('geo_id'):
    lw    = 3.0  if geo_id == BRAZIL else 1.2
    color = '#1565C0' if geo_id == BRAZIL else '#BDBDBD'
    alpha = 1.0  if geo_id == BRAZIL else 0.7
    label = LABELS[geo_id]
    ax.plot(group['release_label'], group['value'],
            marker='o', linewidth=lw, color=color, alpha=alpha,
            markersize=8 if geo_id == BRAZIL else 5, label=label)

    # Anota apenas o Brasil
    if geo_id == BRAZIL:
        for _, row in group.iterrows():
            ax.annotate(f"{row['value']:.2f}×",
                        xy=(row['release_label'], row['value']),
                        xytext=(0, 12), textcoords='offset points',
                        ha='center', fontsize=10, color='#1565C0', fontweight='bold')

ax.axhline(1, color='#78909C', linestyle='--', linewidth=1, label='Média global')
ax.set_ylabel('Índice de uso per capita', fontsize=11)
ax.set_title('Evolução da Adoção de IA per capita — Brasil vs. países comparáveis\n'
             '(Ago 2025 → Nov 2025 → Fev 2026)', fontsize=TITLE_SIZE, pad=12)
ax.legend(loc='upper left', fontsize=9, ncol=2)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'slide0_temporal_trend.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Versão interativa Plotly — mostra todos os países com hover
fig = px.line(
    df_trend,
    x='release_label', y='value', color='country',
    markers=True,
    title='Evolução do Índice de Uso per capita — Ago 2025 → Fev 2026',
    labels={'release_label': 'Período', 'value': 'Índice per capita', 'country': 'País'},
    color_discrete_map={'Brasil': '#1565C0'},
    line_dash_map={'Brasil': 'solid'},
)
fig.add_hline(y=1, line_dash='dot', line_color='gray',
              annotation_text='média global', annotation_position='bottom right')
fig.update_traces(line_width=1.5)
# Destaca Brasil
for trace in fig.data:
    if trace.name == 'Brasil':
        trace.line.width = 4
        trace.marker.size = 10
fig.update_layout(yaxis_title='Índice de uso per capita (1 = média global)')
fig.write_html(OUTPUT_DIR / 'slide0_temporal_trend_interactive.html')
fig.show()

In [ ]:
# Tabela: variação % do Brasil entre primeira e última release
brazil_trend = df_trend[df_trend['geo_id'] == BRAZIL].sort_values('release_date')
v_start = brazil_trend.iloc[0]['value']
v_end   = brazil_trend.iloc[-1]['value']
delta   = (v_end - v_start) / v_start

print('Índice Brasil por período:')
print(brazil_trend[['release_label', 'value']].rename(columns={'value': 'usage_per_capita_index'}).to_string(index=False))
print(f'\nVariação Ago 2025 → Fev 2026: {"+" if delta >= 0 else ""}{delta:.1%}')
print(f'Posição relativa na última release: {"ACIMA" if v_end >= 1 else "ABAIXO"} da média global')

<div style="background:#E3F2FD;padding:14px 20px;border-left:4px solid #1565C0;border-radius:6px;">
<strong>💡 Insight temporal:</strong> Uma linha ascendente do Brasil — especialmente acelerando mais do que Argentina e México —
é a evidência mais forte contra o argumento de bolha. Adoção real não para de crescer.
</div>

---
# 📌 Contexto: o que é o Anthropic Economic Index?

<div style="background:#E3F2FD;padding:20px 24px;border-left:4px solid #1565C0;border-radius:6px;">
<p>A Anthropic mapeou <strong>como o Claude está sendo usado de verdade</strong> por pessoas em todo o mundo — não o que as empresas dizem que vão fazer com IA, mas o que os usuários realmente pedem ao modelo.</p>
<ul>
<li>Dados de semanas de uso reais (3 snapshots: Ago 2025, Nov 2025, Fev 2026)</li>
<li>Cobertura: Claude.ai (Free + Pro) em <strong>dezenas de países</strong></li>
<li>Classificação por tarefas do <strong>O*NET</strong> (taxonomia ocupacional dos EUA)</li>
<li>Métricas: uso per capita, tipo de colaboração, automação vs. augmentação</li>
</ul>
<p><em>É o termômetro mais confiável que temos hoje sobre adoção real de IA no trabalho.</em></p>
</div>

---
# 🌍 Slide 1 — Onde o Brasil está no mapa da adoção?

**Pergunta:** O Brasil usa mais ou menos IA do que a média global, comparado a países similares?

In [ ]:
df_usage = (
    df[
        (df['geography'] == 'country') &
        (df['facet']     == 'country') &
        (df['variable']  == 'usage_per_capita_index') &
        (df['geo_id'].isin(COMPARABLES))
    ]
    .assign(country=lambda x: x['geo_id'].map(LABELS))
    .sort_values('value')
)

fig, ax = plt.subplots(figsize=(10, 4.5))
colors = ['#1565C0' if g == BRAZIL else ('#CFD8DC' if v < 1 else '#90CAF9')
          for g, v in zip(df_usage['geo_id'], df_usage['value'])]
bars = ax.barh(df_usage['country'], df_usage['value'], color=colors, height=0.6)
ax.axvline(1, color='#37474F', linestyle='--', linewidth=1.5, label='Média global = 1')

for bar, val in zip(bars, df_usage['value']):
    ax.text(val + 0.02, bar.get_y() + bar.get_height()/2,
            f'{val:.2f}×', va='center', fontsize=9)

ax.set_xlabel('Índice de uso per capita (média global = 1)', fontsize=11)
ax.set_title('Adoção de IA per capita — Brasil vs. países comparáveis (Ago 2025)', fontsize=TITLE_SIZE, pad=12)
ax.legend(loc='lower right')
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'slide1_usage_index.png', dpi=150, bbox_inches='tight')
plt.show()

<div style="background:#FFF8E1;padding:14px 20px;border-left:4px solid #F9A825;border-radius:6px;">
<strong>💡 Insight:</strong> O índice revela se o Brasil está acima ou abaixo da média global de uso per capita.
Países acima de 1× estão usando IA de forma desproporcional ao seu tamanho — sinal de adoção ativa, não apenas curiosidade.
</div>

---
# 🤖 Slide 2 — Automação ou Augmentação?

**Pergunta:** Os brasileiros estão usando IA para *substituir* trabalho (automação) ou para *ampliar* suas capacidades (augmentação)?

> **Automação** = o modelo executa a tarefa de forma autônoma (diretiva, loop de feedback)  
> **Augmentação** = o humano mantém controle — valida, itera, aprende

In [ ]:
df_auto = (
    df[
        (df['geography'] == 'country') &
        (df['facet']     == 'country') &
        (df['variable'].isin(['automation_pct', 'augmentation_pct'])) &
        (df['geo_id'].isin(COMPARABLES))
    ]
    .assign(
        country=lambda x: x['geo_id'].map(LABELS),
        tipo   =lambda x: x['variable'].map({'automation_pct': 'Automação', 'augmentation_pct': 'Augmentação'})
    )
)

df_pivot = (
    df_auto.pivot_table(index='country', columns='tipo', values='value')
    .reset_index()
    .sort_values('Automação', ascending=False)
)

fig, ax = plt.subplots(figsize=(10, 4.5))
x = range(len(df_pivot))
w = 0.38
ax.bar([i - w/2 for i in x], df_pivot['Automação'],   width=w, label='Automação',   color='#EF5350', alpha=0.9)
ax.bar([i + w/2 for i in x], df_pivot['Augmentação'], width=w, label='Augmentação', color='#42A5F5', alpha=0.9)

ax.set_xticks(list(x))
ax.set_xticklabels(df_pivot['country'], fontsize=10)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
ax.set_title('Como o Claude é usado — Automação vs. Augmentação por País', fontsize=TITLE_SIZE, pad=12)
ax.legend(fontsize=10)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'slide2_auto_vs_aug.png', dpi=150, bbox_inches='tight')
plt.show()

<div style="background:#E8F5E9;padding:14px 20px;border-left:4px solid #2E7D32;border-radius:6px;">
<strong>💡 Insight:</strong> Se a augmentação domina, é um sinal positivo — as pessoas estão usando IA para crescer,
não para terceirizar o pensamento. Isso é o oposto de hype: é uso intencional e produtivo.
</div>

---
# 📋 Slide 3 — O que os brasileiros pedem ao Claude?

**Pergunta:** Quais tarefas profissionais reais o Brasil mais usa com IA?

In [ ]:
df_tasks_br = (
    df[
        (df['geo_id']   == BRAZIL) &
        (df['facet']    == 'onet_task') &
        (df['variable'] == 'onet_task_pct')
    ]
    .nlargest(15, 'value')
    .sort_values('value')
)

fig, ax = plt.subplots(figsize=(11, 6))
bars = ax.barh(df_tasks_br['cluster_name'], df_tasks_br['value'],
               color='#1565C0', alpha=0.85, height=0.65)

for bar, val in zip(bars, df_tasks_br['value']):
    ax.text(val + 0.001, bar.get_y() + bar.get_height()/2,
            f'{val:.1%}', va='center', fontsize=8.5)

ax.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
ax.set_title('Top 15 Tarefas O*NET mais usadas no Brasil', fontsize=TITLE_SIZE, pad=12)
ax.set_xlabel('% do volume de uso no país')
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'slide3_top_tasks_brazil.png', dpi=150, bbox_inches='tight')
plt.show()

<div style="background:#E8F5E9;padding:14px 20px;border-left:4px solid #2E7D32;border-radius:6px;">
<strong>💡 Insight:</strong> Se as tarefas são cognitivamente densas — escrita, análise, código, resolução de problemas —
isso derruba a narrativa de hype. Pessoas estão usando Claude para trabalho real de alto valor, não só para curiosidade.
</div>

---
# 🔬 Slide 4 — O Brasil é diferente?

**Pergunta:** Nas tarefas onde o Brasil mais se destaca, somos parecidos com Argentina e México — ou temos um perfil único?

In [ ]:
df_idx = (
    df[
        (df['geography'] == 'country') &
        (df['facet']     == 'onet_task') &
        (df['variable']  == 'onet_task_pct_index') &
        (df['geo_id'].isin(COMPARABLES))
    ]
    .assign(country=lambda x: x['geo_id'].map(LABELS))
)

top_tasks_br = (
    df_idx[df_idx['geo_id'] == BRAZIL]
    .nlargest(10, 'value')['cluster_name'].tolist()
)

df_heat = (
    df_idx[df_idx['cluster_name'].isin(top_tasks_br)]
    .pivot_table(index='cluster_name', columns='country', values='value')
)

fig, ax = plt.subplots(figsize=(13, 5))
sns.heatmap(
    df_heat, annot=True, fmt='.1f', cmap='RdYlGn',
    center=1, linewidths=0.5, ax=ax,
    cbar_kws={'label': 'Índice (1 = média global)'}
)
ax.set_title(
    'Índice de especialização — Top 10 tarefas do Brasil vs. países comparáveis\n'
    '(verde > 1 = mais do que a média global | vermelho < 1 = menos)',
    fontsize=12, pad=12
)
ax.set_xlabel('')
ax.set_ylabel('')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'slide4_heatmap_specialization.png', dpi=150, bbox_inches='tight')
plt.show()

<div style="background:#FFF8E1;padding:14px 20px;border-left:4px solid #F9A825;border-radius:6px;">
<strong>💡 Insight:</strong> Células em verde escuro mostram onde o Brasil vai além da média global.
Se o Brasil tem verde em tarefas de alta complexidade cognitiva e outros países comparáveis não,
isso é evidência de uma adoção mais madura e direcionada — não hype.
</div>

---
# 💼 Slide 5 — Quais profissões estão mais expostas?

**Pergunta:** Independente de país — quais ocupações o Claude está penetrando mais no mundo real?

In [ ]:
top20 = df_job.nlargest(20, 'observed_exposure').sort_values('observed_exposure')

fig, ax = plt.subplots(figsize=(11, 7))
bars = ax.barh(top20['title'], top20['observed_exposure'],
               color='#7B1FA2', alpha=0.8, height=0.65)

for bar, val in zip(bars, top20['observed_exposure']):
    ax.text(val + 0.003, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=8.5)

ax.set_title('Top 20 Ocupações com Maior Exposição Observada ao Claude', fontsize=TITLE_SIZE, pad=12)
ax.set_xlabel('Índice de exposição observada')
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'slide5_job_exposure.png', dpi=150, bbox_inches='tight')
plt.show()

pct_above = (df_job['observed_exposure'] > 0.1).mean()
median_exp = df_job['observed_exposure'].median()
print(f'Mediana de exposição (todas as ocupações): {median_exp:.3f}')
print(f'Ocupações com exposição > 0.1: {pct_above:.1%}')

<div style="background:#F3E5F5;padding:14px 20px;border-left:4px solid #7B1FA2;border-radius:6px;">
<strong>💡 Insight:</strong> As ocupações mais expostas tendem a ser de trabalho do conhecimento:
analistas, desenvolvedores, escritores, gestores. Isso responde diretamente se a sua área está dentro do mapa da IA.
</div>

---
# 🎯 Slide 6 — Hype ou Realidade?

## Veredito baseado em dados

In [ ]:
usage_br = df[
    (df['geo_id'] == BRAZIL) & (df['facet'] == 'country') &
    (df['variable'] == 'usage_per_capita_index')
]['value'].values[0]

auto_br = df[
    (df['geo_id'] == BRAZIL) & (df['facet'] == 'country') &
    (df['variable'] == 'automation_pct')
]['value'].values[0]

aug_br = df[
    (df['geo_id'] == BRAZIL) & (df['facet'] == 'country') &
    (df['variable'] == 'augmentation_pct')
]['value'].values[0]

n_tasks_br = df[
    (df['geo_id'] == BRAZIL) & (df['facet'] == 'onet_task') &
    (df['variable'] == 'onet_task_pct') & (df['value'] > 0)
].shape[0]

# Variação temporal do Brasil (Ago 2025 → Fev 2026)
trend_vals = (
    df_all[
        (df_all['geo_id']    == BRAZIL) &
        (df_all['facet']     == 'country') &
        (df_all['variable']  == 'usage_per_capita_index')
    ]
    .sort_values('release_date')['value'].values
)
delta_pct = (trend_vals[-1] - trend_vals[0]) / trend_vals[0] if len(trend_vals) >= 2 else 0

fig, axes = plt.subplots(1, 4, figsize=(15, 4))

cards = [
    (f'{usage_br:.2f}×',
     'Uso per capita\nvs. média global',
     '#1565C0' if usage_br >= 1 else '#B71C1C',
     'Acima da média' if usage_br >= 1 else 'Abaixo da média'),
    (f'{"+" if delta_pct >= 0 else ""}{delta_pct:.0%}',
     'Crescimento\nAgo 2025 → Fev 2026',
     '#00695C' if delta_pct >= 0 else '#B71C1C',
     '6 meses de dados'),
    (f'{aug_br:.1%}',
     'Taxa de\nAugmentação',
     '#42A5F5',
     'Humano no controle'),
    (str(n_tasks_br),
     'Tarefas O*NET\ncom uso registrado',
     '#2E7D32',
     'Diversidade real'),
]

for ax, (val, label, color, sub) in zip(axes, cards):
    ax.set_facecolor(color)
    ax.text(0.5, 0.58, val, ha='center', va='center',
            fontsize=26, fontweight='bold', color='white', transform=ax.transAxes)
    ax.text(0.5, 0.28, label, ha='center', va='center',
            fontsize=10, color='white', transform=ax.transAxes)
    if sub:
        ax.text(0.5, 0.10, sub, ha='center', va='center',
                fontsize=8, color='#E3F2FD', style='italic', transform=ax.transAxes)
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)

fig.suptitle('🇧🇷 Brasil — Resumo do uso real de IA', fontsize=TITLE_SIZE, y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'slide6_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()

---
# 🏁 Conclusão — O que os dados dizem?

<div style="background:#0A2540;color:white;padding:28px 32px;border-radius:10px;line-height:1.8;">

<h3 style="color:#4FC3F7;">Hype ou bolha?</h3>

<p>Os dados do Anthropic Economic Index mostram que o Brasil <strong style="color:#4FC3F7;">não é espectador</strong> na adoção de IA.</p>

<ul style="color:#B0BEC5;">
<li>📈 <strong style="color:white;">Adoção per capita:</strong> o índice revela se estamos acima ou abaixo da média — e o ranking entre emergentes importa</li>
<li>📊 <strong style="color:white;">Tendência temporal:</strong> 3 snapshots (Ago 2025 → Fev 2026) mostram se a adoção está crescendo ou estagnando</li>
<li>🧠 <strong style="color:white;">Perfil de uso:</strong> predominância de augmentação indica uso intencional — pessoas que usam IA para pensar melhor, não para terceirizar o pensamento</li>
<li>📋 <strong style="color:white;">Tarefas reais:</strong> o Brasil usa Claude em tarefas de trabalho do conhecimento — análise, escrita, código, resolução de problemas</li>
<li>💼 <strong style="color:white;">Exposição ocupacional:</strong> profissões de analytics, dados e tecnologia estão entre as mais expostas — quem trabalha na área já está dentro do mapa</li>
</ul>

<br>
<p style="color:#4FC3F7;font-size:16px;"><strong>O hype existe — mas os dados mostram que há substância embaixo dele.</strong></p>
<p>A pergunta não é mais "IA vai impactar o Brasil?" — é <strong style="color:white;">"você está do lado de quem usa para crescer ou do lado de quem ainda está esperando para ver?"</strong></p>

</div>

---
## Fontes e Metodologia

| Item | Detalhe |
|------|------|
| **Dataset** | [Anthropic/EconomicIndex](https://huggingface.co/datasets/Anthropic/EconomicIndex) |
| **Releases** | `release_2025_09_15` · `release_2026_01_15` · `release_2026_03_24` |
| **Períodos** | Ago 2025 · Nov 2025 · Fev 2026 |
| **Plataforma** | Claude.ai (Free + Pro) |
| **Taxonomia** | O\*NET occupational tasks (SOC 2019) |
| **Normalização** | `usage_per_capita_index` = `usage_pct` / (pop_país / pop_total) para releases raw |
| **Análise** | Python · pandas · matplotlib · plotly |

> *Os dados refletem uso de claude.ai — não incluem uso via API corporativa, que pode amplificar os números em contextos enterprise.*